In [ ]:
import pandas as pd
import sqlalchemy as sa 

In [ ]:
#to be filled by user input
selected_year = 0

cargo_2018 = pd.read_csv('2018\\cargodesc\\ams__cargodesc_2018__202001290000_part_1.csv')
header_2018 = pd.read_csv('2018\\header\\ams__header_2018__202001290000_part_1.csv')
tariff_2018 = pd.read_csv('2018\\tariff\\ams__tariff_2018__202001290000_part_0.csv')
shipper_2018 = pd.read_csv('2018\\shipper\\ams__shipper_2018__202001290000_part_0.csv')

cargo_2019 = pd.read_csv('2019\\cargodesc\\ams__cargodesc_2019__202001080000_part_0.csv')
header_2019 = pd.read_csv('2019\\header\\ams__header_2019__202001080000_part_0.csv')
tariff_2019 = pd.read_csv('2019\\tariff\\ams__tariff_2019__202001080000_part_0.csv')
shipper_2019 = pd.read_csv('2019\\shipper\\ams__shipper_2019__202001080000_part_0.csv')

cargo_2020 = pd.read_csv('2020\\cargodesc\\ams__cargodesc_2020__202009291500_part_0.csv')
header_2020 = pd.read_csv('2020\\header\\ams__header_2020__202009291500_part_0.csv')
tariff_2020 = pd.read_csv('2020\\tariff\\ams__tariff_2020__202009291500_part_0.csv')
shipper_2020 = pd.read_csv('2020\\shipper\\ams__shipper_2020__202009291500_part_0.csv')

In [ ]:
def get_year():
    global selected_year
    while selected_year not in [2018, 2019, 2020]:
        try:
            selected_year = int(input("Please enter a year: 2018, 2019, or 2020: "))
        except ValueError:
            print("invalid input, please enter a valid year.")
get_year()

In [ ]:
def assign_dataframe(selected_year):
    if selected_year == 2018:
        cargo_df = cargo_2018
        header_df = header_2018
        tariff_df = tariff_2018
        shipper_df = shipper_2018
    elif selected_year == 2019:
        cargo_df = cargo_2019
        header_df = header_2019
        tariff_df = tariff_2019
        shipper_df = shipper_2019
    elif selected_year == 2020:
        cargo_df = cargo_2020
        header_df = header_2020
        tariff_df = tariff_2020
        shipper_df = shipper_2020
    return cargo_df, header_df, tariff_df, shipper_df

In [ ]:
'''print("Cargo 2018 columns:", cargo_2018.columns.tolist())
print("Header 2018 columns:", header_2018.columns.tolist())
print("Tariff 2018 columns:", tariff_2018.columns.tolist())
print("Shipper 2018 columns:", shipper_2018.columns.tolist())

print("\nCargo 2019 columns:", cargo_2019.columns.tolist())
print("Header 2019 columns:", header_2019.columns.tolist())
print("Tariff 2019 columns:", tariff_2019.columns.tolist())
print("Shipper 2019 columns:", shipper_2019.columns.tolist())

print("\nCargo 2020 columns:", cargo_2020.columns.tolist())
print("Header 2020 columns:", header_2020.columns.tolist())
print("Tariff 2020 columns:", tariff_2020.columns.tolist())
print("Shipper 2020 columns:", shipper_2020.columns.tolist())'''

In [ ]:
# Clean and prepare the selected year data for 2NF

def clean_data(selected_year):
    cargo_df, header_df, tariff_df, shipper_df = assign_dataframe(selected_year)

    # Shipment table (from header): key = identifier
    shipment_columns = [
        'identifier', 'carrier_code', 'vessel_name', 'vessel_country_code', 'port_of_unlading',
        'estimated_arrival_date', 'actual_arrival_date', 'weight', 'weight_unit'
    ]
    header_cleaned_df = header_df[shipment_columns].drop_duplicates().copy()
    header_cleaned_df['weight_combined'] = header_cleaned_df['weight'].astype(str) + ' ' + header_cleaned_df['weight_unit']
    header_cleaned_df = header_cleaned_df.drop(columns=['weight', 'weight_unit'])

    # Shipper table: key = identifier
    shipper_columns = ['identifier', 'shipper_party_name', 'city', 'country_code']
    shipper_norm_df = shipper_df[shipper_columns].drop_duplicates().copy()

    # Cargo description table: key = (identifier, container_number, description_sequence_number)
    cargo_columns = [
        'identifier', 'container_number', 'description_sequence_number',
        'piece_count', 'description_text'
    ]
    cargo_norm_df = cargo_df[cargo_columns].copy()

    # Tariff table: key = (identifier, container_number, description_sequence_number)
    tariff_columns = [
        'identifier', 'container_number', 'description_sequence_number',
        'harmonized_number', 'harmonized_value', 'harmonized_weight',
        'harmonized_weight_unit'
    ]
    tariff_clean_df = tariff_df[tariff_columns].copy()
    tariff_clean_df['harmonized_weight_combined'] = (
        tariff_clean_df['harmonized_weight'].astype(str) + ' ' +
        tariff_clean_df['harmonized_weight_unit']
    )
    tariff_clean_df = tariff_clean_df.drop(columns=['harmonized_weight', 'harmonized_weight_unit'])

    # Countries table: unique country codes from shipper and vessel
    countries_shipper = set(shipper_norm_df['country_code'].dropna().unique())
    countries_vessel = set(header_cleaned_df['vessel_country_code'].dropna().unique())
    all_countries = countries_shipper.union(countries_vessel)
    countries_df = pd.DataFrame({'country_code': sorted(list(all_countries))})

    return header_cleaned_df, shipper_norm_df, cargo_norm_df, tariff_clean_df, countries_df


header_cleaned_df, shipper_norm_df, cargo_norm_df, tariff_clean_df, countries_df = clean_data(selected_year)

print("Shipment table shape:", header_cleaned_df.shape)
print("Shipper table shape:", shipper_norm_df.shape)
print("Cargo table shape:", cargo_norm_df.shape)
print("Tariff table shape:", tariff_clean_df.shape)
print("Countries table shape:", countries_df.shape)

print("\nSample shipment weight_combined:")
print(header_cleaned_df['weight_combined'].head())

print("\nSample tariff harmonized_weight_combined:")
print(tariff_clean_df['harmonized_weight_combined'].head())

In [ ]:
header_cleaned_df

In [ ]:
import os

# Create bronze directory if it doesn't exist
os.makedirs('bronze', exist_ok=True)

# Save cleaned dataframes to CSV for use in silver layer
header_cleaned_df.to_csv(f'bronze/shipment_{selected_year}_cleaned.csv', index=False)
shipper_norm_df.to_csv(f'bronze/shipper_{selected_year}_cleaned.csv', index=False)
cargo_norm_df.to_csv(f'bronze/cargo_{selected_year}_cleaned.csv', index=False)
tariff_clean_df.to_csv(f'bronze/tariff_{selected_year}_cleaned.csv', index=False)
countries_df.to_csv(f'bronze/countries_{selected_year}.csv', index=False)

print("Cleaned data saved to bronze/ directory.")